#  1. Preparação e Upload dos Dados
Carregue os arquivos CSV disponibilizados na pasta Inputs como Volume no Data Lakehouse do Databricks.


In [0]:
# Importações e criação da sessão Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Criar sessão spark
spark = SparkSession.builder.appName("AtividadePraticaSilverToGold").getOrCreate()

In [0]:
# Configuração de catálogo, schemas e caminho da landing zone
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"Landing: {landing_path}")

catalog: olist_medallion
bronze_schema: olist_medallion.bronze
silver_schema: olist_medallion.silver
gold_schema: olist_medallion.gold
Landing: /Volumes/olist_medallion/bronze/landing


In [0]:
# Criação do catálogo, schemas e volume da landing zone
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

DataFrame[]

In [0]:
# Validação dos arquivos esperados na landing zone
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar {landing_path}. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos estão na landing zone.")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos estão na landing zone.
  - cotacao_dolar.csv (0.2 KB)
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


# Parte 2 — Consagração do Negócio (Camada Gold)

Nesta etapa você construirá os Data Marts consolidados, a modelagem dimensional (Star Schema) para consumo do
time de BI. Utilize exibições de DataFrames em tela (display()) para responder às perguntas de negócio solicitadas.
Nenhuma biblioteca de visualização de gráficos é necessária.


### Entrega 1 — Modelagem Dimensional (Star Schema)

Construa as seguintes tabelas, utilizando chaves substitutas (Surrogate Keys) geradas via
monotonically_increasing_id(), row_number() ou função de hash (sha2):

### gold.dim_movies

In [0]:
# Importação dos dados movie_info da camada silver
filmes = spark.table(f"{silver_schema}.movie_info")

w = Window.orderBy("id_filme")

# Criação da tabela dim_movies
dim_movies = (
    filmes
    .withColumn("sk_movie_id", F.row_number().over(w).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string").alias("id_filme"),
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        F.expr("try_cast(duracao_minutos as int)").alias("duracao_minutos"),
        "idioma_original",
        "status_filme",
        "sinopse",
    )
)

dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_movies")

display(dim_movies.orderBy("sk_movie_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,14564,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
2,32471,Mixtape,2021-12-03,2021,94,en,Lançado,null
3,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly
4,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
5,38700,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
6,42018,The Horse Thief,2019-03-19,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
7,42330,Monkey Magic,2018-09-22,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
8,43074,Ghostbusters,2016-07-14,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat."
9,45033,20 Seconds of Joy,2018-01-01,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear."
10,46983,The Song of Styrene,2022-05-23,2022,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.


### gold.fact_movies_performance


In [0]:
# Importação dos dados movie_financials da camada silver
financials = spark.table(f"{silver_schema}.movie_financials")

# Importação dos dados movie_metrics da camada silver
metrics = spark.table(f"{silver_schema}.movie_metrics")

# Criação da tabela fact_movies_performance
fact_movies_performance = (
    financials
    .join(metrics, on="id_filme", how="inner")
    .join(dim_movies, on="id_filme", how="inner")
    .select(
        "id_filme",
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal (18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal (18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal (18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal (18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal (18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal (18,2)").alias("lucro_brl"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb")
)

fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_movies_performance")

display(fact_movies_performance.orderBy("id_filme").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_filme,sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
14564,1,25000000.00,83080890.00,58080890.00,135000000.00,448636806.00,313636806.00,24.584,4.966,2375,null,46286
32471,2,null,null,null,null,null,null,8.929,7.064,118,6.6,4617
38258,3,7500000.00,null,null,40500000.00,null,null,null,3.161,28,null,null
38492,4,null,null,null,null,null,null,2.98,7.1,null,7.8,212
38700,5,90000000.00,426505244.00,336505244.00,486000000.00,2303128317.60,1817128317.60,46.619,7.139,7570,6.5,199420
42018,6,null,null,null,null,null,null,3.403,6.63,27,6.8,1750
42330,7,null,null,null,null,null,null,1.561,10.0,1,6.4,34
43074,8,144000000.00,229147509.00,85147509.00,777600000.00,1237396548.60,459796548.60,40.052,5.371,5976,null,260417
45033,9,337200.00,null,null,1820880.00,null,null,0.6,8.0,2,null,154
46983,10,null,null,null,null,null,null,0.693,8.5,2,7.0,1209


### gold.dim_genres

In [0]:
# Importação dos dados generos da camada silver
generos = spark.table(f"{silver_schema}.generos")

w = Window.orderBy("id_filme")

# Criação da tabela dim_genre
dim_genres = (
    generos
    .withColumn("sk_genre_id", F.row_number().over(w).cast("bigint"))
    .select("sk_genre_id", "genero")
)

dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_genres")

display(dim_genres.orderBy("sk_genre_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_genre_id,genero
1,Horror
2,Comedy
3,Family
4,Music
5,Crime
6,Action
7,Thriller
8,Drama
9,Fantasy
10,Animation


### gold.dim_people


In [0]:
# Importação dos dados das entidades da camada silver
people = spark.table(f"{silver_schema}.tb_pessoas_empresas")

w = Window.orderBy("id_filme")

# Criação da tabela dim_people
dim_people = (
    people
    .where(F.col("tipo_entidade") != "Produtora")
    .withColumn("sk_person_id", F.row_number().over(w).cast("bigint"))
    .select("sk_person_id",
             F.col("entidade").alias("nome_pessoa"), 
             F.col("tipo_entidade").alias("tipo_pessoa"))
)

dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_people")

display(dim_people.orderBy("sk_person_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_person_id,nome_pessoa,tipo_pessoa
1,Aimee Teegarden,Ator
2,Matilda Lutz,Ator
3,Johnny Galecki,Ator
4,Vincent D'Onofrio,Ator
5,Laura Slade Wiggins,Ator
6,Bonnie Morgan,Ator
7,Jacob Estes,Roteirista
8,Akiva Goldsman,Roteirista
9,David Loucka,Roteirista
10,Chuck David Willis,Ator


### gold.dim_companies

In [0]:
w = Window.orderBy("id_filme")

# Criação da tabela dim_people
dim_companies = (
    people
    .where(F.col("tipo_entidade") == "Produtora")
    .withColumn("sk_company_id", F.row_number().over(w).cast("bigint"))
    .select("sk_company_id", 
             F.col("entidade").alias("nome_produtora"))
)

dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_companies")

display(dim_companies.orderBy("sk_company_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_company_id,nome_produtora
1,Vertigo Entertainment
2,Waddieish Claretrap
3,Paramount
4,Parkes+MacDonald Image Nation
5,Jim Wedaa Productions
6,Maven Screen Media
7,Columbia Music Video
8,Columbia Pictures
9,Overbrook Entertainment
10,2.0 Entertainment


### gold.dim_reviews

In [0]:
# Importação dos dados do movie_reviews da camada silver
review = spark.table(f"{silver_schema}.movie_reviews")

w = Window.orderBy("id_filme")

# Criação da tabela dim_people
dim_reviews = (
    review
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
    .join(dim_movies, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(w).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_reviews")

display(dim_reviews.orderBy("sk_review_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,2,1,8.3
2,3,2,5.15
3,4,1,2.5
4,5,1,0.1
5,10,1,0.2
6,15,1,4.7
7,16,1,4.9
8,24,1,9.3
9,26,1,1.2
10,28,1,8.2


### gold.bridge_movie_genre

In [0]:
# Criação da tabela bridge_movie_genre
bridge_movie_genre = (
    dim_movies
    .join(generos, on="id_filme", how="inner")
    .join(dim_genres, on="genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
)

bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_genre")

display(bridge_movie_genre.orderBy("sk_movie_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,sk_genre_id
1,132936
1,132951
1,132971
1,132961
1,132939
1,132942
1,132938
1,132935
1,132959
1,132933


### gold.bridge_movie_person

In [0]:
# Utiliza uma tabela auxiliara com a coluna nome_pessoa renomeada para realizar o join posteriormente 
dim_people_aux = dim_people.withColumnRenamed("nome_pessoa", "entidade")

# Criação da tabela bridge_movie_genre
bridge_movie_person = (
    dim_movies
    .join(people, on="id_filme", how="inner")
    .join(dim_people_aux, on="entidade", how="inner")
    .select("sk_movie_id", "sk_person_id")
)

bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_person")

display(bridge_movie_person.orderBy("sk_movie_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,sk_person_id
1,246569
1,345388
1,652181
1,611577
1,555813
1,658437
1,301556
1,192463
1,571744
1,40787


### gold.bridge_movie_company

In [0]:
# Utiliza uma tabela auxiliara com a coluna nome_produtora renomeada para realizar o join posteriormente 
dim_companies_aux = dim_companies.withColumnRenamed("nome_produtora", "entidade")

# Criação da tabela bridge_movie_genre
bridge_movie_company = (
    dim_companies_aux
    .join(people, on="entidade", how="inner")
    .join(dim_movies, on="id_filme", how="inner")
    .select("sk_movie_id", "sk_company_id")
)

bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_company")

display(bridge_movie_company.orderBy("sk_movie_id").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,sk_company_id
1,44942
1,67035
1,91546
1,97990
1,53883
1,82919
1,40614
1,29798
1,88942
1,27871


# 4. Desafio de Analytics

Para validar sua modelagem Gold, resolva as consultas abaixo, as quais abordam perguntas que poderiam ser
recorrentes em pautas do dia a dia de um Analista de Dados

### Perguntas de Negócio:


### 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW olist_medallion.gold.vw_q1_receita_total_brl AS
    SELECT round(sum(receita_brl), 2) AS receita_total_brl
    FROM olist_medallion.gold.fact_movies_performance
""")

display(spark.table(f"{gold_schema}.vw_q1_receita_total_brl"))

receita_total_brl
874082719080.00


### 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW olist_medallion.gold.vw_q2_top5_popularidade AS
    SELECT m.titulo AS Titulo, f.popularidade AS Popularidade, m.data_lancamento AS Data_Lancamento
    FROM olist_medallion.gold.fact_movies_performance f
    JOIN olist_medallion.gold.dim_movies m
    ON f.sk_movie_id = m.sk_movie_id
    ORDER BY f.popularidade DESC
    LIMIT 5
""")

display(spark.table(f"{gold_schema}.vw_q2_top5_popularidade"))

Titulo,Popularidade,Data_Lancamento
blue beetle,2994.357,2023-08-16
Gran Turismo,2680.593,2023-08-09
La Fellinette,2020.0,2020-10-18
The Fear Footage 2: Curse of the Tape,2019.0,2020-03-12
wwe survivor series 2018,2018.0,2018-11-18


### 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW olist_medallion.gold.vw_q3_filmes_por_genero AS
    SELECT d_g.genero AS Genero, COUNT(*) AS Qtd_Filmes
    FROM olist_medallion.gold.dim_genres d_g
    JOIN olist_medallion.gold.bridge_movie_genre b_m_g
    ON d_g.sk_genre_id = b_m_g.sk_genre_id
    GROUP BY genero
    ORDER BY Qtd_Filmes DESC
""")

display(spark.table(f"{gold_schema}.vw_q3_filmes_por_genero"))

Genero,Qtd_Filmes
Drama,941318910
Documentary,351547124
Comedy,306097695
Thriller,90160738
Horror,85345976
Romance,49651602
Action,31174660
Crime,18978586
Animation,17618250
TV Movie,13719255


### 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking
(RANK()).


In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW olist_medallion.gold.vw_q4_top10_receita AS
    WITH ranking AS (
    SELECT m.titulo,
            f.receita_usd,
            f.receita_brl,
            RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
    FROM olist_medallion.gold.fact_movies_performance f
    JOIN olist_medallion.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
    )
    SELECT titulo, receita_usd, receita_brl, posicao
    FROM ranking
    WHERE posicao <= 10
    ORDER BY posicao, titulo
""")

display(spark.table(f"{gold_schema}.vw_q4_top10_receita"))

titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,15120000000.00,1
Avatar: The Way of Water,2320250281.00,12529351517.40,2
AVENGERS: INFINITY WAR,2052415039.00,11083041210.60,3
spider-man: no way home,1921847111.00,10377974399.40,4
The Lion King,1663075401.00,8980607165.40,5
Top Gun: Maverick,1488732821.00,8039157233.40,6
Barbie,1428545028.00,7714143151.20,7
The Super Mario Bros. Movie,1355725263.00,7320916420.20,8
Black Panther,1349926083.00,7289600848.20,9
Star Wars: The Last Jedi,1332698830.00,7196573682.00,10


### 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW olist_medallion.gold.vw_q5_ator_mais_participacoes_2anos AS
    WITH data_max AS (
        SELECT m.data_lancamento
        FROM olist_medallion.gold.dim_movies m
        WHERE m.status_filme = 'Lançado'
        AND data_lancamento <= current_date()
        ORDER BY m.data_lancamento DESC
        LIMIT 1
    )
    SELECT p.nome_pessoa AS Ator, COUNT(*) AS Qtd_Participacao
    FROM olist_medallion.gold.dim_people p
    JOIN olist_medallion.gold.bridge_movie_person b_m_p
    ON p.sk_person_id = b_m_p.sk_person_id
    WHERE p.tipo_pessoa = 'Ator' AND p.nome_pessoa != 'Nenhum'
    AND b_m_p.sk_movie_id IN (
        SELECT sk_movie_id
        FROM olist_medallion.gold.dim_movies m
        WHERE m.data_lancamento >= (SELECT m.data_lancamento - INTERVAL 2 YEAR FROM data_max)
    )
    GROUP BY Ator
    ORDER BY Qtd_Participacao DESC
    LIMIT 1
""")

display(spark.table(f"{gold_schema}.vw_q5_ator_mais_participacoes_2anos"))

Ator,Qtd_Participacao
Eric Roberts,37636


### 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW olist_medallion.gold.vw_q6_produtora_maior_lucro_5anos  AS
    WITH data_max AS (
        SELECT m.data_lancamento
        FROM olist_medallion.gold.dim_movies m
        WHERE m.status_filme = 'Lançado'
        AND data_lancamento <= current_date()
        ORDER BY m.data_lancamento DESC
        LIMIT 1
    )
    SELECT c.nome_produtora AS Produtora, SUM(f.lucro_brl) AS Lucro_BRL, SUM(f.lucro_usd) AS Lucro_USD
    FROM olist_medallion.gold.dim_companies c
    JOIN olist_medallion.gold.bridge_movie_company b_m_c
    ON c.sk_company_id = b_m_c.sk_company_id
    JOIN olist_medallion.gold.dim_movies m
    ON b_m_c.sk_movie_id = m.sk_movie_id
    JOIN olist_medallion.gold.fact_movies_performance f
    ON m.sk_movie_id = f.sk_movie_id
    WHERE f.lucro_brl > 0
    AND b_m_c.sk_movie_id IN (
        SELECT sk_movie_id
        FROM olist_medallion.gold.dim_movies m
        WHERE m.data_lancamento >= (SELECT m.data_lancamento - INTERVAL 5 YEAR FROM data_max)
    )
    GROUP BY Produtora
    ORDER BY Lucro_BRL DESC
    LIMIT 1
""")

display(spark.table(f"{gold_schema}.vw_q6_produtora_maior_lucro_5anos"))

Produtora,Lucro_BRL,Lucro_USD
Universal Pictures,14589875117138.40,2701828725396.00
